[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Vision and Language - Image & Text Embedding

This notebooks shows how to use Large Vision Model for meta data extraction from a football game frame.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/10/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0130DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing
import imageio.v3 as iio

# Deep Learning
from transformers import AutoTokenizer, AutoModel

# Miscellaneous
import json
import os
import random

# Typing
from typing import Callable, Dict, Literal, Optional, Tuple, Union
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython
from IPython.display import JSON, Markdown, display

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages


In [ ]:
# General Auxiliary Functions

def ParseEnvFile( fileName: str = '.env', *, filePath: str = '.', keyValSep: str = '=' ) -> Dict[str, str]:
    # Read the file and parse it into a dictionary
    dEnv = {}
    with open(os.path.join(filePath, fileName), 'r') as hF:
        for line in hF:
            if line.startswith('#') or not line.strip():
                continue
            key, value = line.strip().split(keyValSep, 1)
            dEnv[key]  = value.strip()
    return dEnv

## Motivation

Let a dataset be composed of 10 _classes_ with 15 images per class.  
The task is to classify the images.

<!-- ![](https://i.imgur.com/DCe552s.jpeg) -->
![](https://i.postimg.cc/YCWMLrRp/image.png)

How can the problem be solved?  
Can it be solved **without** a specific training?

## Multi Modal Encoders / Multi Modal Embedding Models

A _Modality_ is a type of data: _Text_, _Image_, _Audio_ or _Video_.  
An _Encoder_ converts an input into a vector of numbers.  
This vector is called an _Embedding_. It represents learned features of the input.

An _Image Encoder_ maps an image to an embedding.  
A _Text Encoder_ maps a chunk of text (Word / Sentence / Passage) to an embedding.

### A Shared Embedding Space

In order to generate a shared embedding space 2 constraints must be forced:

 - The embedding space of the text and images must match in their _dimension_.
 - The representations must be _aligned_.

Related images and text should have similar embeddings.  
During training, matching image and text pairs are encouraged to have higher similarity.  
Mismatched pairs are encouraged to have lower similarity.  
The encoders learn a _Joint / Shared Embedding Space_.

* <font color='brown'>(**#**)</font> A well known example is the [OpenAI CLIP Model](https://openai.com/index/clip).  
  Its name stands for [Contrastive Language Image Pre Training](https://en.wikipedia.org/wiki/Contrastive_Language%E2%80%93Image_Pre-training).  
* <font color='brown'>(**#**)</font> This notebook covers the evolution of `CLIP` (2021) -> `SigLIP` (2023) -> `SigLIP 2` (2025).

## The Contrastive Language Image Pre Training (CLIP) Model

CLIP learns from matching _Image / Text_ pairs.  
An image encoder processes the images. A text encoder processes their descriptions.  
The task during training is to find which image matches which text.

![](https://i.imgur.com/LjZfI95.png)
<!-- ![](https://i.postimg.cc/8CVhvChq/image.png) -->

### Training: Compare Every Pair

The training is based on having a dataset of Image and Text pairs.  
For instance, an image of a sunflower and the text _"a photo of a sunflower"_.  

The training is driving the model to learn a _similarity matrix_:

 - Take a batch of 4 matching pairs.  
 - Compare **every image** with **every text**.  
   This gives a $4 \times 4$ _Similarity Matrix_.  
 - The loss function maximizes the value on diagonal and minimizes the values off diagonal.

The comparison is based on _Cosine Similarity_.

The following table illustrates an example:

| Image / Text     | "an airplane"    | "a helicopter"   | "a sunflower"   | "a pizza"       |
|------------------|------------------|------------------|-----------------|-----------------|
| Airplane Image   | **0.82**         | 0.43             | 0.09            | 0.05            |
| Helicopter Image | 0.38             | **0.79**         | 0.07            | 0.04            |
| Sunflower Image  | 0.06             | 0.08             | **0.85**        | 0.12            |
| Pizza Image      | 0.03             | 0.05             | 0.14            | **0.88**        |

The loss function drives teh model to learn such matrices per batch.

### Learn in Both Directions

Per batch there are 2 interleaved stages.  


 - Image to Text: 
    - Pick an image.
    - Calculate similarity to each text.
    - Scale the result by a learned parameter.
    - Apply classification loss with the _Cross Entropy Loss_.
 - Text to Image: 
    - Pick a text.
    - Calculate similarity to each image.
    - Scale the result by a learned parameter.
    - Apply classification loss with the _Cross Entropy Loss_.

The loss is a weighted sum of both directions.

* <font color='red'>(**?**)</font> Why might an airplane image have some similarity to the text _"a helicopter"_?
* <font color='brown'>(**#**)</font> One may consider a loss vs. the Identity matrix.  
  Yet it will force perpendicularity off diagonal which is a strong geometrical constraint and might be wrong contextually (See question above).
* <font color='brown'>(**#**)</font> The table contains similarities, not probabilities.  
  Softmax probabilities depend on the other candidates and the score scaling.
* <font color='brown'>(**#**)</font> Different pairs in a batch may still describe similar content.  
  Treating all off diagonal pairs as mismatches is a training assumption and a weakness of the model.

In [ ]:
# Parameters

# Data
dataDirName = 'ImgZeroShotCls' #<! https://huggingface.co/datasets/Royi/DataSets -> `ImgZeroShotCls.zip`
lImgFileExt = ['.jpg', '.png']

# Model
modelName = 'siglip2'

## The Task: Classify Images into 10 Classes with No Training

<!-- ![](https://i.imgur.com/DCe552s.jpeg) -->
![](https://i.postimg.cc/YCWMLrRp/image.png)

* <font color='red'>(**?**)</font> Given a _CLIP_ like model, how would you solve the problem above?

## Generate / Load Data

 - Download the file `ImgZeroShotCls.zip` from https://huggingface.co/datasets/Royi/DataSets.
 - Unpack it into `FixelCourses\DataSets`.
 - Verify images in `FixelCourses\DataSets\ImgZeroShotCls`.

In [ ]:
# Generate / Load Data

dataDirPath = os.path.join(DATA_FOLDER_PATH, dataDirName)

# The folder is organized:
#  - ImgZeroShotCls
#     - <Class Name 001>
#       - <Image Files>
#     - <Class Name 002>
#       - <Image Files>

dData = {}

for clsName in sorted(os.listdir(dataDirPath)):
    clsDirPath = os.path.join(dataDirPath, clsName)
    if not os.path.isdir(clsDirPath):
        continue

    for imgFileName in sorted(os.listdir(clsDirPath)):
        imgFilePath = os.path.join(clsDirPath, imgFileName)
        if os.path.isfile(imgFilePath) and os.path.splitext(imgFileName)[1].lower() in lImgFileExt:
            dData[os.path.join(clsName, imgFileName)] = clsName

lImgFile   = list(dData)
lClsName   = sorted(set(dData.values()))
numSamples = len(dData)
numClasses = len(lClsName)

print(f'Number of images: {numSamples}')
print(f'Number of classes: {numClasses}')

### Plot the Data

In [ ]:
# Plot the Data

hF, vHa = plt.subplots(nrows = 2, ncols = 5, figsize = (12, 5))
vHa = vHa.flat

for clsName, hA in zip(lClsName, vHa):
    lClsImgFile = [imgFileName for imgFileName in lImgFile if dData[imgFileName] == clsName]
    imgFileName = random.choice(lClsImgFile)
    imgFilePath = os.path.join(dataDirPath, imgFileName)

    tI = iio.imread(imgFilePath)

    hA.imshow(tI)
    hA.set_title(clsName)
    hA.axis('off')

hF.tight_layout()

* <font color='red'>(**?**)</font> Is the whole image needed in order to accomplish the task? Think about optimizing the cost.

## The Hugging Face Transformers Package

[Hugging Face Transformers](https://huggingface.co/docs/transformers) is an open source Python package.  
It provides implementations of many models for _Text_, _Images_, _Audio_ and combinations of them.  
Its main contribution is a **common interface** for loading, using and fine tuning _Pre Trained Models_.

Instead of implementing each architecture ourselves, we can reuse existing models and learned weights.  
This makes it easier to try different models and build on their capabilities.

### Main Concepts

 - Model  
   The neural network. It maps prepared inputs to outputs such as embeddings, scores or generated text.
 - Checkpoint  
   A saved configuration and learned weights.  
   It specifies a particular version of a model.
 - Tokenizer  
   Converts text into token IDs. The model then maps these IDs to learned embeddings.
 - Processor  
   Prepares inputs in the format expected by the model.  
   For image and text models, it combines image preprocessing with tokenization.
 - Auto Classes:  
   Classes such as `AutoModel`, `AutoTokenizer` and `AutoProcessor`.  
   They select the appropriate implementation for a given checkpoint.
 - Pipeline  
   A high level interface for supported tasks.  
   It combines preprocessing, model inference and output processing.

### The Hugging Face Hub

The [Hugging Face Hub](https://huggingface.co/models) hosts model repositories.  
A repository may contain weights, configuration files, preprocessing settings and documentation.  
A model is usually identified by an `organization/model-name` string.

The method `from_pretrained()` loads the relevant files from a repository or a local folder.  
Downloaded files are cached for reuse. The model can then run locally on a CPU or a GPU.

### In This Notebook

The notebooks demonstrates loading a pre trained image and text embedding model and its matching processor.  
The processor prepares the images and text. The model produces their embeddings.  
Then it compare the embeddings to classify images without training a new classifier.

* <font color='red'>(**?**)</font> Without the `transformers` package, how would you push a string of text into a Pytorch model?
* <font color='brown'>(**#**)</font> Each processor is associated with a specific checkpoint as the model.  
  Different checkpoints may require different tokenization and image preprocessing.

## The SigLIP Model

The CLIP model has few weaknesses:
 - Coupled Comparisons  
   The probability of one pair depends on the other candidates in its row or column.  
   The loss cannot evaluate each pair independently.
 - Global Normalization  
   Softmax needs normalization over all candidates in a row or column.  
   This complicates memory efficient training across multiple devices.
 - Small Batch Performance  
   CLIP's Softmax loss benefits from many mismatched pairs in a batch.  
   It can be less effective when only small batches are practical.

The _SigLIP_ models are evolution of the _CLIP_ model.
SigLIP keeps the same architecture of an image encoder and a text encoder.  
The main change is the _Loss Function_.  
It replaces the loss over a batch of candidates into a per pair loss: **Find the best match from the candidates** → **Score each pair match**.

 - Independent Pair Loss  
   Each image and text pair becomes a binary classification task: _Match / No Match_.  
   Its cosine similarity is scaled and shifted by learned parameters.  
   A _Sigmoid_ converts this score into a value between 0 and 1.
 - No Global Normalization  
   Each pair contributes its own loss. No row or column normalization is needed.  
   Pair losses can be computed in chunks and combined.  
   This enables more memory efficient training across multiple devices.
 - Better Small Batch Performance  
   The [SigLIP paper](https://arxiv.org/abs/2303.15343) reports better results at smaller batch sizes than the Softmax loss.  
   The method also scales to large batches. Large batches can still be useful.

For matching training pairs, the target is **1**. For mismatched pairs, it is **0**.  
The _Binary Cross Entropy Loss_ penalizes incorrect predictions for each pair.  
The pair losses are combined to update both encoders.

* <font color='brown'>(**#**)</font> The Sigmoid scores do not need to sum to 1 across a row or column.  
  One pair's score does not need to decrease just because another pair receives a higher score.
* <font color='brown'>(**#**)</font> SigLIP still treats off diagonal training pairs as mismatches.  
  Removing Softmax does not solve the problem of related pairs being labeled as negatives.
* <font color='brown'>(**#**)</font> The _SigLIP 2_ model mainly focuses on improving the training process: Masked patch prediction, Region level supervision, Multi language, Flexible resolution, etc..

In [ ]:
# Model



In [ ]:
# Load Image

imgIdx = 123
imgFileName = lImgFile[imgIdx]
imgFilePath = os.path.join(dataDirPath, imgFileName)
tI = iio.imread(imgFilePath)

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tI)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Pre Process Image

numRows, numCols, numChannels = tI.shape

tX = tI[:(numRows // 2), :(numCols // 2), :]
tX = sp.ndimage.zoom(tX, (0.5, 0.5, 1.0))

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tX)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Prompt

promptMsg = "The input image is a frame from a video of a football game. The task is to extract the match period, the time. the score, the stoppage time and the time within the stoppage time."

In [ ]:
# Answer

answerMsg = oSessionProse.Prompt(promptMsg, tX)
display(Markdown(answerMsg))

* <font color='brown'>(**#**)</font> Modern day LLM's are so good they "predict" the intention.

## Structured Session

In [ ]:
# Structured Response Schema

systemMsg = (
    "Extract match metadata from the scoreboard in this football frame. Follow the response schema. "
    "Use only visible information. Do not guess. Use null for missing or unreadable values. "
    "Set `has_time` to false when no readable match clock is visible. "
    "Read scores from left to right or top to bottom. "
    "Distinguish extra time from stoppage time. Record announced added time and elapsed stoppage time separately. "
    "Set `confidence_level` to 'None' if nothing can be extracted, 'Low' for uncertain readings or 'High' for clear readings."
)

dSchema = {
    "type": "object",
    "properties": {
        "match_period": {
            "type": ["string", "null"],
            "enum": ["1st Half", "2nd Half", "1st Extratime", "2nd Extratime"],
            "description": "Match period identified from visible information, or null when unavailable.",
        },
        "has_time": {
            "type": "boolean",
            "description": "Whether a match clock is visible in the frame.",
        },
        "time_mm_ss": {
            "type": ["string", "null"],
            "description": "Visible match time in MM:SS format, or null when unavailable.",
            "pattern": r"^\d{2}:\d{2}$",
        },
        "confidence_level": {
            "type": "string",
            "enum": ["None", "Low", "High"],
            "description": "Overall extraction confidence. None: no requested metadata can be extracted. Low: uncertain extraction. High: extracted values are clear.",
        },
        "stoppage_time_referee": {
            "type": ["string", "null"],
            "description": "Visible stoppage time decided by the referee in +X format, or null when unavailable.",
            "pattern": r"^\+\d+$",
        },
        "stoppage_time_actual": {
            "type": ["string", "null"],
            "description": "Visible actual stoppage time in MM:SS (e.g. `02:19`) or M:SS (e.g. `2:19`) format, or null when unavailable.",
            "pattern": r"^\d{1,2}:\d{2}$",
        },
        "first_team_score": {
            "type": ["integer", "null"],
            "description": "Visible score of the first team, or null when unavailable.",
            "minimum": 0,
            "maximum": 99,
        },
        "second_team_score": {
            "type": ["integer", "null"],
            "description": "Visible score of the second team, or null when unavailable.",
            "minimum": 0,
            "maximum": 99,
        },
    },
    "required": [
        "match_period", "has_time", "time_mm_ss", "confidence_level", "stoppage_time_referee",
        "stoppage_time_actual", "first_team_score", "second_team_score",
    ],
    "additionalProperties": False,
}

promptMsg = "Extract the match metadata from the scoreboard in this football frame following the structured response schema."

In [ ]:
# Structured Response Section

oSessionStructured = LLMSessionStructured(dEnv[apiKey], modelName, dSchema = dSchema, systemMsg = systemMsg, contextWindowBudget = contextWindowSize, reasoningEffort = reasoningEffort, valTemp = modelTemp)

In [ ]:
# Answer

answerMsg = oSessionStructured.Prompt(promptMsg, tX)
display(Markdown(f'```json\n{json.dumps(answerMsg, indent = 4, ensure_ascii = False)}\n```'))

In [ ]:
hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tX)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Close Sessions

oSessionProse.Close()
oSessionStructured.Close()

* <font color='green'>(**@**)</font> Build a class and a script to process the whole folder. MNind the token usage. Save to output folder with the output result for analysis.